In [1]:
import pandas as pd
import numpy as np
import xarray as xr
from datetime import datetime
from ICONProcessor import ICONDataGrid

data_dir = '../data_in/'
data_out_dir = '../data_out/'

In [2]:
def get_file(avg=False, sfc=False, pl=False):
    if avg:
        avg_str = '_30min'
    else:
        avg_str = ''
    
    if pl:
        suffix = 'pl__uv'
    elif sfc:
        suffix = 'ml__sfc'
    else:
        suffix = 'ml__lowest5'
    
      
    file = f'{project_root}/{output_dir}/{exp}_LES{avg_str}_51m_{suffix}.nc'
    return file

def format_datetime64(val, format_str='%Y-%m-%d %H:%M:%S'):
    dt = pd.to_datetime(str(val))
    return dt.strftime(format_str)

## Script settings

In [7]:
scope = 'H2'

project_root = '/work/bb1461/'
exp = 'v370_2030'

In [8]:
if scope == 'H2':
    scope_name = 'HEFEX II'
    output_dir = 'cdo_scripts/output_hefex2/'
    experiment_id =f"hefex2/{exp}/exp_R3B15_51m"
elif scope == 'H3':
    scope_name = 'HEFEX III'
    output_dir = 'cdo_scripts/output_hefex3/'
    experiment_id = f"hefex3/{exp}/exp_R3B15_51m"
else:
    print('Invalid scope!')

loc_file = data_dir + f'{scope}_Coords_w_cells.csv'

## Load and prepare location dimension

In [9]:
df_locs = pd.read_csv(loc_file).sort_values('topo')
df_locs = df_locs[~df_locs['cell_id'].isna()]
df_locs

,New Name,Type,Lat,Lon,Comment,Alt,cell_id,topo
0,A255,AWS EC,46.810911,10.784094,MOMAA4. on debris,2556.0,34960.0,2654.089111
1,A260,AWS EC,46.809083,10.779727,MOMAA3,2605.0,37345.0,2690.791504
10,D262,AWS Met,46.808336,10.778370,TLOGGER,2623.0,37360.0,2722.822510
2,A267R,AWS EC,46.805415,10.776431,MOMAA9. Swiss line,2671.0,37442.0,2757.178711
3,A267L,AWS EC,46.806711,10.773102,MOMAA10. Swiss line,2672.0,37425.0,2762.057373
4,A267,AWS EC,46.805913,10.775135,MOMAA8. Swiss line,2671.0,37433.0,2762.127930
11,D269,AWS Wind,46.804296,10.773574,AWS,2692.0,37287.0,2792.991455
12,D273,AWS Met,46.801794,10.771454,TLOGGER,2725.0,35704.0,2798.971191
6,A275L,AWS EC,46.800647,10.768919,Grenoble,2741.0,35706.0,2802.588623
5,A275R,AWS EC,46.799891,10.771788,Grenoble,2749.0,36058.0,2808.865967


In [6]:
dim_loc = df_locs['New Name']
dim_loc = dim_loc.replace('Hintereis','STHE')

dim_loc_lat = np.array(df_locs['Lat'])
dim_loc_lon = df_locs['Lon']
dim_loc_cid = df_locs['cell_id'].astype(int)
dim_loc_ele = df_locs['topo']

## Load and prepare ICON data for locations

In [7]:
def create_data_array(ds, icon_var, height, name, time_dim='time'):
    # get time from dataset
    time = [ICONDataGrid.convert_float_dt(f) for f in ds.time.values]
    time_naive = [dt.replace(tzinfo=None) for dt in time]

    # get data_in for variable (all cell IDs)
    data = ds[icon_var].values[:, height, np.array(dim_loc_cid-1)]

    # create DataArray
    xr_data = xr.DataArray(
        data,
        coords={
            time_dim: (time_dim, time_naive, {"long_name": "Timestamp in UTC"}),
            "location": ("location", dim_loc, {"long_name": "Location name from field campaign"}),
            "lat": ("location", dim_loc_lat, {"long_name": "Latitude", "units": "degrees_north"}),
            "lon": ("location", dim_loc_lon, {"long_name": "Longitude", "units": "degrees_east"}),
            "cid": ("location", dim_loc_cid, {"long_name": "ICON Cell ID of location"}),
            "ele": ("location", dim_loc_ele, {"long_name": "Elevation on ICON grid", "units": "meters"})
        },
        dims=[time_dim, "location"],
        name=name,
        attrs={
            "long_name": ds[icon_var].long_name,
            "standard_name": ds[icon_var].standard_name,
            "units": ds[icon_var].units
        }
    )
    return xr_data

In [8]:
# surface values
ds = xr.open_dataset(get_file(sfc=True))
xr_t_2m  = create_data_array(ds, 't_2m',  0, 't_2m')
xr_qv_2m = create_data_array(ds, 'qv_2m', 0, 'qv_2m')

# values from lowest model level 
ds = xr.open_dataset(get_file(sfc=False))
xr_t_5m  = create_data_array(ds, 'temp', -1, 't_5m')
xr_qv_5m = create_data_array(ds, 'qv',   -1, 'qv_5m')
xr_u_5m  = create_data_array(ds, 'u',    -1, 'u_5m')
xr_v_5m  = create_data_array(ds, 'v',    -1, 'v_5m')

# values from 500hPa pressure level
ds = xr.open_dataset(get_file(pl=True))
xr_u_500hPa  = create_data_array(ds, 'u', 0, 'u_500hPa')
xr_v_500hPa  = create_data_array(ds, 'v', 0, 'v_500hPa')

# 30min avg values
ds = xr.open_dataset(get_file(avg=True))
xr_u_5m_avg  = create_data_array(ds, 'u', -1, 'u_5m_avg', 'time_avg')
xr_v_5m_avg  = create_data_array(ds, 'v', -1, 'v_5m_avg', 'time_avg')

## Build dataset

In [9]:
# Combine DataArrays
ds_out = xr.Dataset({
    't_2m' : xr_t_2m,
    't_5m' : xr_t_5m,
    'qv_2m': xr_qv_2m,                     
    'qv_5m': xr_qv_5m, 
    'u_5m' : xr_u_5m,
    'v_5m' : xr_v_5m,
    'u_500hPa': xr_u_500hPa,                     
    'v_500hPa': xr_v_500hPa, 
    'u_5m_avg' : xr_u_5m_avg,
    'v_5m_avg' : xr_v_5m_avg,
})

# clean time series (fill gaps and remove duplicates)
ds_out = ds_out.sortby("time")
ds_out = ds_out.sel(time=~ds_out.indexes["time"].duplicated())
time_full = pd.date_range(ds_out.time.min().values, ds_out.time.max().values, freq="1h")
ds_out = ds_out.reindex(time=time_full)

ds_out = ds_out.sortby("time_avg")
ds_out = ds_out.sel(time_avg=~ds_out.indexes["time_avg"].duplicated())
time_full = pd.date_range(ds_out.time_avg.min().values, ds_out.time_avg.max().values, freq="30min")
ds_out = ds_out.reindex(time_avg=time_full)

# Add global attributes
ds_out.attrs = {
    "title": f"Sliced ICON-LES dataset at 51m resolution for {scope_name} locations",
    "institution": "Humboldt-Universität zu Berlin",
    "source": "icon-2024.10 (https://gitlab.dkrz.de/icon/icon-model.git)",
    "history": f"Created {datetime.now().strftime('%Y-%m-%d')}",
    "contact": "alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-6761)",
    "experiment_id": experiment_id,
    "StartTime": format_datetime64(ds_out.time.values[0]),
    "StopTime": format_datetime64(ds_out.time.values[-1]),
    "comment": f"Based on glacier outlines from OGGM SSP370, year 2030"
}

# Write
output_file = data_out_dir + f'{scope}_ICON_{exp}_stations.nc'
ds_out.to_netcdf(output_file)
print(f'Output file saved as {output_file}.')

Output file saved as H2_ICON_v370_2030_stations.nc.


In [10]:
ds_out

<xarray.Dataset> Size: 692kB
Dimensions:   (time: 739, location: 19, time_avg: 1477)
Coordinates:
  * time      (time) datetime64[ns] 6kB 2023-08-15 ... 2023-09-14T18:00:00
  * location  (location) object 152B 'A255' 'A260' 'D262' ... 'A309' 'D316'
  * time_avg  (time_avg) datetime64[ns] 12kB 2023-08-15 ... 2023-09-14T18:00:00
    lat       (location) float64 152B 46.81 46.81 46.81 ... 46.79 46.79 46.8
    lon       (location) float64 152B 10.78 10.78 10.78 ... 10.75 10.75 10.74
    cid       (location) int64 152B 34960 37345 37360 ... 35803 38570 71618
    ele       (location) float64 152B 2.654e+03 2.691e+03 ... 3.185e+03
Data variables:
    t_2m      (time, location) float32 56kB 276.8 276.7 276.6 ... 274.9 274.3
    t_5m      (time, location) float32 56kB 282.8 282.4 282.2 ... 276.3 275.7
    qv_2m     (time, location) float32 56kB 0.005287 0.005279 ... 0.005653
    qv_5m     (time, location) float32 56kB 0.007996 0.007879 ... 0.006259
    u_5m      (time, location) float32 56kB 0.4233 0.3258 ... 0.4799 0.3683
    v_5m      (time, location) float32 56kB -0.8704 -0.9073 ... -0.5097 -0.06999
    u_500hPa  (time, location) float32 56kB 0.6796 0.6855 ... -9.046 -9.143
    v_500hPa  (time, location) float32 56kB 2.76 2.761 2.762 ... -1.288 -1.236
    u_5m_avg  (time_avg, location) float32 112kB 0.4233 0.3258 ... 0.2072 0.1052
    v_5m_avg  (time_avg, location) float32 112kB -0.8704 -0.9073 ... 0.06425
Attributes:
    title:          Sliced ICON-LES dataset at 51m resolution for HEFEX II lo...
    institution:    Humboldt-Universität zu Berlin
    source:         icon-2024.10 (https://gitlab.dkrz.de/icon/icon-model.git)
    history:        Created 2026-04-10
    contact:        alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-946...
    experiment_id:  hefex2/v370_2030/exp_R3B15_51m
    StartTime:      2023-08-15 00:00:00
    StopTime:       2023-09-14 18:00:00
    comment:        Based on glacier outlines from OGGM SSP370, year 2030